In [1]:
import os
os.environ["PATH"] += os.pathsep + r"C:\Program Files\Graphviz\bin"
import matplotlib.pyplot as plt
%matplotlib inline
import random
import math

In [2]:
class Value:
    def __init__(self,data, _children=(), _op='',label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda : None
        self._prev = set(_children)
        self._op = _op
        self.label = label
        
    def __repr__(self):
        return f"Value(data = {self.data})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self,other), '+')
        def _backward():
            self.grad += out.grad*1.0
            other.grad += out.grad*1.0
        out._backward = _backward
        return out
    
    def __mul__(self,other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self,other), '*')
        
        def _backward():
            self.grad += out.grad*other.data
            other.grad += out.grad*self.data
            
        out._backward = _backward
        return out
    
    def __pow__(self, other):
        assert isinstance(other,(int, float)), "sadece int ve float tip geçerlidir"
        out = Value(self.data**other, (self, ), f"**{other}")

        def _backward():
            self.grad += other * (self.data**(other-1))*out.grad
            
        out._backward = _backward
        return out   
    
    def __truediv__(self, other):
        return self*other**-1
    def __neg__(self):
        return self*-1
    
    def __sub__(self, other):
        return self + (-other)
     
    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += out.data*out.grad
        out._backward = _backward
        return out
    
    def __radd__(self, other):
        return self + other
    
    def __rmul__(self, other):
        return self * other
    
    def tanh(self):
        x = self.data
        t = (math.exp(2*x)-1)/(math.exp(2*x)+1)
        out = Value(t, (self, ), 'tanh')
        
        def _backward():
            self.grad += (1-t**2) * out.grad
        
        out._backward = _backward
        return out
    
    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
                
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

In [3]:
from graphviz import Digraph

def trace(root):
    # builds a set of all nodes and edges in a graph
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
    
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        # for any value in the graph, create a rectangular ('record') node for it
        dot.node(name = uid, label = "{ %s |  data %.4f | grad %4f}" % (n.label, n.data, n.grad ), shape='record')
        if n._op:
            # if this value is a result of some operation, create an op node for it
            dot.node(name = uid + n._op, label = n._op)
            # and connect this node to it
            dot.edge(uid + n._op, uid)
            
    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
        
    return dot

In [12]:
class Neuron:
    def __init__(self,nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))
        
    def __call__(self,x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out
    
    def parameters(self):
        return self.w + [self.b]
    
class Layer:
    def __init__(self,nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
        
    def __call__(self,x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
    
    
class MLP:
    def __init__(self,nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
        
    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()] 

In [14]:
n = MLP(3, [4,4,1])

# inputs
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
# ture outputs
ys = [1.0, -1.0, -1.0, 1.0]


In [15]:
for k in range(20):
    #forward pass
    y_pred = [n(x) for x in xs]
    loss = sum((yout-ygt)**2 for ygt, yout in zip(ys, y_pred))
    
    # backward pass
    for p in n.parameters():
        p.grad = 0.0
        
    loss.backward()
    
    # update
    for p in n.parameters():
        p.data += -0.1 * p.grad
        
    print(f"{k}. {loss.data}")

0. 5.55065817285651
1. 4.471596187602896
2. 3.8869005037395774
3. 3.815512803243781
4. 4.188702134226221
5. 4.450957989631003
6. 3.8991420745177603
7. 2.8993901937450604
8. 2.905475927839945
9. 2.710775975940098
10. 2.703366307596509
11. 2.6917166911286037
12. 2.6845758985619423
13. 2.6733082520362252
14. 2.662297983702515
15. 2.6433352798588223
16. 2.61326405223354
17. 2.5433571451961465
18. 2.328200412243289
19. 1.6938180085779944


In [16]:
n.parameters()

[Value(data = 0.973654415833651),
 Value(data = 0.5554490972430136),
 Value(data = -0.49923841520106255),
 Value(data = 0.6068425316137885),
 Value(data = 1.0087767223311601),
 Value(data = -0.25276305730407256),
 Value(data = -1.289808641053956),
 Value(data = 0.13040853500425714),
 Value(data = -0.1432269363397229),
 Value(data = 1.0631111393326749),
 Value(data = 0.4338005982658388),
 Value(data = 0.31069969480267184),
 Value(data = -0.5337865508554114),
 Value(data = 0.2755308962979412),
 Value(data = 0.34506986015623337),
 Value(data = -0.8301430349308091),
 Value(data = 0.6642331552424865),
 Value(data = -0.5664347456844075),
 Value(data = -0.16197812813951518),
 Value(data = -0.764677797917521),
 Value(data = -0.45951780885561483),
 Value(data = 0.8606253380990209),
 Value(data = 1.076372379886904),
 Value(data = -0.27655253435570626),
 Value(data = -0.039590159643103306),
 Value(data = -0.3262061994835733),
 Value(data = -0.3297273176257373),
 Value(data = -1.0841414418872584),